In [0]:
%run ./config

In [0]:
dbutils.secrets.list("kv-scope")

[SecretMetadata(key='sp-secret')]

In [0]:
import zipfile
import os

# Step 1: Copy zip from RAW to local
dbutils.fs.cp("abfss://raw@datalakecars.dfs.core.windows.net/used_cars/used-car-dataset.zip", "file:///tmp/used-car-dataset.zip")

# Step 2: Unzip locally
os.makedirs("/tmp/used_cars_extracted/", exist_ok=True)
with zipfile.ZipFile("/tmp/used-car-dataset.zip", 'r') as z:
    z.extractall("/tmp/used_cars_extracted/")
    print("Extracted:", z.namelist())

# Step 3: Copy only v2 to Bronze
dbutils.fs.cp("file:///tmp/used_cars_extracted/used_cars_dataset_v2.csv", "abfss://bronze@datalakecars.dfs.core.windows.net/used_cars/used_cars_dataset_v2.csv")

print("v2 Saved to Bronze ✅")

Extracted: ['used_car_dataset.csv', 'used_cars_dataset_v2.csv']
v2 Saved to Bronze ✅


In [0]:
df1 = spark.read.option("header", "true").csv("abfss://bronze@datalakecars.dfs.core.windows.net/used_cars/used_car_dataset.csv")
df2 = spark.read.option("header", "true").csv("abfss://bronze@datalakecars.dfs.core.windows.net/used_cars/used_cars_dataset_v2.csv")

print("df1 rows:", df1.count(), "| columns:", len(df1.columns))
print("df2 rows:", df2.count(), "| columns:", len(df2.columns))
print("\ndf1 columns:", df1.columns)
print("\ndf2 columns:", df2.columns)

df1 rows: 9582 | columns: 11
df2 rows: 14993 | columns: 11

df1 columns: ['Brand', 'model', 'Year', 'Age', 'kmDriven', 'Transmission', 'Owner', 'FuelType', 'PostedDate', 'AdditionInfo', 'AskPrice']

df2 columns: ['Brand', 'model', 'Year', 'Age', 'kmDriven', 'Transmission', 'Owner', 'FuelType', 'PostedDate', 'AdditionInfo', 'AskPrice']
